In [1]:
import numpy as np
import time
import mujoco
import mujoco.viewer
import matplotlib.pyplot as plt

import osqp
import scipy.sparse as sp

In [2]:
xml_path = '/home/frlab/mj_opt/xmls/husky_ur5e/scene_husky_ur5e_3finger.xml'

model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)

ee_site_name = "attachment_site"
ee_site_id = data.site(ee_site_name).id

In [3]:
def draw_axes(user_scn, pos, rot_mat, size=0.1):
    R = rot_mat.reshape(3, 3)
    colors = [
        [1, 0, 0, 1],  # X - 빨강
        [0, 1, 0, 1],  # Y - 초록
        [0, 0, 1, 1],  # Z - 파랑
    ]
    for i in range(3):
        if user_scn.ngeom >= user_scn.maxgeom:
            break
        end = pos + R[:, i] * size
        mujoco.mjv_initGeom(
            user_scn.geoms[user_scn.ngeom],
            mujoco.mjtGeom.mjGEOM_ARROW,
            np.zeros(3), np.zeros(3), np.zeros(9),
            np.array(colors[i], dtype=np.float32)
        )
        mujoco.mjv_connector(
            user_scn.geoms[user_scn.ngeom],
            mujoco.mjtGeom.mjGEOM_ARROW, 0.005,
            pos, end
        )
        user_scn.ngeom += 1

#### 유틸리티

In [4]:
def so32Vec(so3mat):
    omega = np.array([so3mat[2, 1], so3mat[0, 2], so3mat[1, 0]])
    return omega


def Vec2so3(omega):
    w1, w2, w3 = np.array(omega).flatten()
    so3mat = np.array([[0, -w3, w2],
                       [w3, 0, -w1],
                       [-w2, w1, 0]])
    return so3mat


def MatrixLog3(R):
    cos_theta = np.clip((np.trace(R) - 1) / 2.0, -1, 1)
    theta = np.arccos(cos_theta)

    if theta < 1e-6:
        return np.zeros((3, 3))
    elif abs(theta - np.pi) < 1e-6:
        RpI = R + np.eye(3)
        col = np.argmax(np.linalg.norm(RpI, axis=0))
        w = RpI[:, col] / np.linalg.norm(RpI[:, col])
        return Vec2so3(w * theta)
    else:
        so3mat = theta / (2 * np.sin(theta)) * (R - np.array(R).T)
        return so3mat

#### 자코비안 분리

In [5]:
def seperate_jacobian(J_full):
    J_base  = J_full[:, 0:6]    # (6 x 6) free joint (베이스)
    J_wheel = J_full[:, 6:10]   # (6 x 4) 휠
    J_arm   = J_full[:, 10:16]  # (6 x 6) UR5e
    J_hand  = J_full[:, 16:22]  # (6 x 6) 핸드
    return J_base, J_arm


def reduce_base_jacobian(J_base):
    # 비홀로노믹 차륜 베이스: [v_x_body, w_z_body] 두 자유도만 사용
    # 0번 열 = base x선속도, 5번 열 = base yaw 각속도
    return J_base[:, [0, 5]]

#### 차륜 구동 변환

In [6]:
def diff_drive_controller(L, r, mobile_ang, mobile_lin):
    v_left  = (mobile_lin - mobile_ang * L / 2) / r
    v_right = (mobile_lin + mobile_ang * L / 2) / r
    return v_left, v_right, v_left, v_right

#### WBC QP (osqp)

결정변수 `x = [v_x_base, w_z_base, dq_arm(6)]` (총 8차원)

$$\min_{x}\; \tfrac12 \|J x - V_{des}\|_W^2 + \tfrac12 x^T R x \quad \text{s.t.}\; l \le x \le u$$

In [7]:
class WBCQPSolver:
    def __init__(self, n_base=2, n_arm=6,
                 w_task=1.0,
                 w_reg_base=1e-2, w_reg_arm=1e-3,
                 v_base_max=0.5, w_base_max=1.0, dq_arm_max=1.5):
        self.nx = n_base + n_arm
        self.W = w_task * np.eye(6)
        self.R = np.diag([w_reg_base] * n_base + [w_reg_arm] * n_arm)

        self.lb = np.array([-v_base_max, -w_base_max] + [-dq_arm_max] * n_arm)
        self.ub = np.array([ v_base_max,  w_base_max] + [ dq_arm_max] * n_arm)

        self.A = sp.eye(self.nx, format='csc')
        self.prob = osqp.OSQP()

        # P sparsity 패턴을 '상삼각 전체 dense'로 setup 시점에 고정한다.
        # nnz = nx*(nx+1)/2. CSC 순서: 각 열 c 에 대해 행 0..c
        P0_dense = np.triu(np.ones((self.nx, self.nx)))
        self._P_pattern = sp.csc_matrix(P0_dense)
        q0 = np.zeros(self.nx)
        self.prob.setup(self._P_pattern, q0, self.A, self.lb, self.ub,
                        verbose=False, warm_starting=True,
                        eps_abs=1e-6, eps_rel=1e-6)

        # CSC 순서의 (row, col) 인덱스를 미리 뽑아둔다.
        # numpy fancy indexing 에 그대로 쓰기 위해 배열로.
        self._csc_rows = self._P_pattern.indices.copy()
        self._csc_cols = np.repeat(
            np.arange(self.nx),
            np.diff(self._P_pattern.indptr)
        )

    def solve(self, J_base_red, J_arm, V_desired):
        J = np.hstack([J_base_red, J_arm])           # (6, nx)
        P = J.T @ self.W @ J + self.R                # (nx, nx) 대칭 PSD
        q = -J.T @ self.W @ V_desired                # (nx,)

        # 상삼각을 CSC 순서로 뽑아 Px 에 넘긴다.
        P_upper = np.triu(P)
        P_data = P_upper[self._csc_rows, self._csc_cols]

        self.prob.update(Px=P_data, q=q)
        res = self.prob.solve()
        if res.info.status_val not in (1, 2):  # solved / solved_inaccurate
            print(f"[QP] status: {res.info.status}")
            return np.zeros(self.nx)
        return res.x

#### Twist 오차

In [8]:
def calc_twist_error(current_pos, current_ori, target_pos, target_ori):
    R_current = current_ori.reshape(3, 3)
    pos_err = target_pos - current_pos

    R_tgt = target_ori.reshape(3, 3)
    R_err = R_tgt @ R_current.T
    ori_err = so32Vec(MatrixLog3(R_err))

    return np.concatenate((pos_err, ori_err))

#### 파라미터

In [9]:
Kp = 5.0 * np.eye(6)
V_ff = np.zeros(6)

# 모바일 베이스
L = 0.5708  # m, 휠 트랙
r = 0.1778  # m, 휠 반지름

dt = 0.005
sim_dt = model.opt.timestep

qp = WBCQPSolver(
    n_base=2, n_arm=6,
    w_task=1.0,
    w_reg_base=1e-2, w_reg_arm=1e-3,
    v_base_max=0.5, w_base_max=1.0, dq_arm_max=1.5,
)

#### 시뮬레이션

In [10]:
with mujoco.viewer.launch_passive(model, data, show_left_ui=False, show_right_ui=False) as viewer:
    viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False

    sim_time = 0.0
    duration = 1000.0
    step_start = time.time()

    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)

    init_ee_pos = data.site(ee_site_id).xpos.copy()
    init_ee_ori = data.site(ee_site_id).xmat.copy()

    target_pos = init_ee_pos + np.array([0.0, 0.0, 0.5])
    target_ori = init_ee_ori.copy().reshape(3, 3) + np.array([0.1, 0.1, 0.0])

    while viewer.is_running() and sim_time < duration:
        # ---- 현재 상태 ----
        current_pos = data.site(ee_site_id).xpos
        current_ori = data.site(ee_site_id).xmat

        J_full = np.zeros((6, model.nv))
        mujoco.mj_jacSite(model, data, J_full[:3], J_full[3:], ee_site_id)
        J_base, J_arm = seperate_jacobian(J_full)
        J_base_red = reduce_base_jacobian(J_base)

        twist_error = calc_twist_error(current_pos, current_ori, target_pos, target_ori)
        V_desired = V_ff + Kp @ twist_error

        # ---- WBC QP ----
        x_opt = qp.solve(J_base_red, J_arm, V_desired)
        mobile_lin = float(x_opt[0])
        mobile_ang = float(x_opt[1])
        desired_dq = x_opt[2:]

        # ---- 휠 변환 ----
        v_left, v_right, _, _ = diff_drive_controller(L, r, mobile_ang, mobile_lin)

        # ---- 명령 적용 ----
        current_q = data.qpos[11:17]
        desired_q = current_q + desired_dq * dt
        data.ctrl[0:4] = [v_left, v_right, v_left, v_right]
        data.ctrl[4:10] = desired_q
        data.ctrl[10:22] = np.zeros(12)
        data.qfrc_applied[10:16] = data.qfrc_bias[10:16]

        mujoco.mj_step(model, data)
        sim_time += sim_dt

        # ---- 시각화 ----
        viewer.user_scn.ngeom = 0
        draw_axes(viewer.user_scn, target_pos, target_ori, size=0.2)
        draw_axes(viewer.user_scn, current_pos, current_ori.flatten(), size=0.2)
        viewer.sync()

        elapsed = time.time() - step_start
        print(f"{elapsed*1000:6.2f}ms  | err={np.linalg.norm(twist_error):.3f}  "
              f"v_b={mobile_lin:+.2f} w_b={mobile_ang:+.2f}  dq_max={np.max(np.abs(desired_dq)):.2f}")

        remaining = sim_dt - elapsed
        if remaining > 0:
            time.sleep(remaining)
        step_start = time.time()

  2.82ms  | err=0.500  v_b=-0.15 w_b=+0.00  dq_max=1.50
  1.23ms  | err=0.500  v_b=-0.15 w_b=+0.00  dq_max=1.50
  0.74ms  | err=0.500  v_b=-0.15 w_b=+0.00  dq_max=1.50
  0.72ms  | err=0.500  v_b=-0.15 w_b=+0.00  dq_max=1.50
  0.62ms  | err=0.500  v_b=-0.15 w_b=+0.00  dq_max=1.50
  0.75ms  | err=0.500  v_b=-0.15 w_b=+0.00  dq_max=1.50
  0.58ms  | err=0.500  v_b=-0.14 w_b=+0.00  dq_max=1.50
  0.58ms  | err=0.500  v_b=-0.14 w_b=-0.00  dq_max=1.50
  0.57ms  | err=0.500  v_b=-0.14 w_b=-0.00  dq_max=1.50
  0.57ms  | err=0.500  v_b=-0.14 w_b=-0.00  dq_max=1.50
  0.57ms  | err=0.500  v_b=-0.14 w_b=-0.00  dq_max=1.50
  0.73ms  | err=0.500  v_b=-0.14 w_b=-0.00  dq_max=1.50
  0.62ms  | err=0.500  v_b=-0.14 w_b=-0.01  dq_max=1.50
  0.61ms  | err=0.500  v_b=-0.14 w_b=-0.01  dq_max=1.50
  0.61ms  | err=0.500  v_b=-0.13 w_b=-0.01  dq_max=1.50
  0.60ms  | err=0.501  v_b=-0.13 w_b=-0.01  dq_max=1.50
  0.74ms  | err=0.501  v_b=-0.13 w_b=-0.01  dq_max=1.50
  0.64ms  | err=0.501  v_b=-0.13 w_b=-0.01  dq_m